In [5]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import sum, count, when, col, desc

# Initialize Spark Session
spark = SparkSession.builder \
    .appName("CapstoneEnrollmentAnalysis") \
    .getOrCreate()

In [6]:
# 1. Loading Data
enrollment_data = [
    (1, 101, "2026-01-10"), (2, 101, "2026-01-12"), (3, 102, "2026-01-15"),
    (4, 103, "2026-01-20"), (5, 101, "2026-01-22"), (6, 102, "2026-01-25"),
    (7, 104, "2025-04-14")
]
progress_data = [
    (1, 100), (2, 45), (3, 100), (4, 10), (5, 0), (6, 100), (7,98)
]
courses_data = [
    (101, "Python Basics"), (102, "Advanced Spark"), (103, "Data Modeling"), (104, "Statistices for Data Science")
]


df_enroll = spark.createDataFrame(enrollment_data, ["student_id", "course_id", "enroll_date"])
df_progress = spark.createDataFrame(progress_data, ["student_id", "completion_pct"])
df_courses = spark.createDataFrame(courses_data, ["course_id", "course_name"])


In [7]:
# 2. Join Tables

df_joined = df_enroll.join(df_progress, "student_id", "inner") \
                     .join(df_courses, "course_id", "inner")


In [10]:

# 3. Group Aggregations

analysis_df = df_joined.groupBy("course_name").agg(
    count("student_id").alias("total_enrolled"),
    sum(when(col("completion_pct") == 100, 1).otherwise(0)).alias("completed_count"),
    sum(when(col("completion_pct") < 20, 1).otherwise(0)).alias("dropout_count")
)

# Calculate Rates
final_report = analysis_df.withColumn(
    "completion_rate", (col("completed_count") / col("total_enrolled")) * 100
)


In [11]:
# 4. Outputs

print("--- Course Performance Report ---")
final_report.orderBy(desc("completion_rate")).show()

print("--- High Dropout Risk Courses ---")
final_report.orderBy(desc("dropout_count")).select("course_name", "dropout_count").show()

--- Course Performance Report ---
+--------------------+--------------+---------------+-------------+-----------------+
|         course_name|total_enrolled|completed_count|dropout_count|  completion_rate|
+--------------------+--------------+---------------+-------------+-----------------+
|      Advanced Spark|             2|              2|            0|            100.0|
|       Python Basics|             3|              1|            1|33.33333333333333|
|Statistices for D...|             1|              0|            0|              0.0|
|       Data Modeling|             1|              0|            1|              0.0|
+--------------------+--------------+---------------+-------------+-----------------+

--- High Dropout Risk Courses ---
+--------------------+-------------+
|         course_name|dropout_count|
+--------------------+-------------+
|       Python Basics|            1|
|       Data Modeling|            1|
|Statistices for D...|            0|
|      Advanced Spark